<a href="https://colab.research.google.com/github/tanveerratul34/Practice-/blob/main/AV_CAV_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
STEP 1 — Verify the string stability threshold  h >= sqrt(2/kp)

This script does three things, in increasing order of convincingness:

  PART A: Simulate one platoon and show the L2 norms of spacing error
          growing (or shrinking) car by car.
  PART B: Inject sinusoids at many frequencies and measure |G(jw)|
          empirically. Compare against the analytic transfer function.
  PART C: Sweep the time gap h and show the peak gain crossing 1.0
          exactly at the predicted threshold sqrt(2/kp).

If PART C's numbers match, you have proved the theorem in your own code.

Model assumptions (deliberately matched to the derivation):
  - no actuator lag (a_i = u_i)          -> so theory applies exactly
  - PD control on CTG spacing error
  - homogeneous platoon
Add lag later; the threshold will move up, which is the point.
"""

import numpy as np

# ---------------------------------------------------------------- parameters
KP   = 0.8          # proportional gain  [1/s^2]
KD   = 1.2          # derivative gain    [1/s]
S0   = 2.0          # standstill gap     [m]
L    = 5.0          # vehicle length     [m]
V_EQ = 25.0         # equilibrium speed  [m/s]
DT   = 0.01         # timestep           [s]

H_CRIT = np.sqrt(2.0 / KP)      # <-- THE PREDICTION


# ------------------------------------------------------- analytic G(s)
def G_magnitude(omega, h, kp=KP, kd=KD):
    """|G(jw)| for  G(s) = (kd*s + kp) / ((1+h*kd)s^2 + (kd+h*kp)s + kp)"""
    jw = 1j * omega
    num = kd * jw + kp
    den = (1 + h * kd) * jw**2 + (kd + h * kp) * jw + kp
    return np.abs(num / den)


def G_peak(h, kp=KP, kd=KD):
    """Worst-case gain over all frequencies = ||G||_inf"""
    w = np.logspace(-3, 2, 4000)
    return G_magnitude(w, h, kp, kd).max()


# ------------------------------------------------------- the simulator
def simulate(h, n_veh=10, t_end=120.0, pert="pulse", omega=None, amp=1.0):
    """
    Platoon of n_veh followers behind a leader.
    Returns the spacing-error time series e[k, i] for each follower.

    The controller is PD on the CTG error. Because a_i = u_i and
    e_dot = dv - h*a_i, the control law is implicit:
        u = kp*e + kd*(dv - h*u)   ->   u = (kp*e + kd*dv) / (1 + kd*h)
    """
    n_steps = int(t_end / DT)
    t = np.arange(n_steps) * DT

    # --- leader speed profile
    if pert == "pulse":
        v_lead = V_EQ + amp * np.exp(-((t - 20.0) ** 2) / (2 * 1.5 ** 2))
    elif pert == "sine":
        v_lead = V_EQ + amp * np.sin(omega * t)
        v_lead[t < 5.0] = V_EQ            # let it settle first
    else:
        raise ValueError(pert)

    # --- initial condition: perfect equilibrium
    gap_eq = S0 + h * V_EQ
    x = np.array([-i * (gap_eq + L) for i in range(n_veh + 1)], dtype=float)
    v = np.full(n_veh + 1, V_EQ)

    e_hist = np.zeros((n_steps, n_veh))

    for k in range(n_steps):
        v[0] = v_lead[k]
        u = np.zeros(n_veh + 1)

        for i in range(1, n_veh + 1):
            s  = x[i - 1] - x[i] - L
            dv = v[i - 1] - v[i]
            e  = s - S0 - h * v[i]
            u[i] = (KP * e + KD * dv) / (1.0 + KD * h)
            e_hist[k, i - 1] = e

        x[0] += v[0] * DT
        v[1:] += u[1:] * DT
        x[1:] += v[1:] * DT

    return t, e_hist


def L2(sig):
    """L2 norm of a signal: sqrt(integral of e^2 dt)"""
    return np.sqrt(np.sum(sig ** 2) * DT)


# =============================================================== PART A
def part_A():
    print("=" * 68)
    print("PART A — spacing error propagating along the platoon")
    print("=" * 68)
    print(f"kp = {KP}, kd = {KD}   ->   predicted threshold h* = "
          f"sqrt(2/kp) = {H_CRIT:.4f} s\n")

    for h in [0.8, H_CRIT, 2.5]:
        _, e = simulate(h, n_veh=8)
        norms = [L2(e[:, i]) for i in range(e.shape[1])]
        ratios = [norms[i] / norms[i - 1] for i in range(1, len(norms))]
        status = "AMPLIFYING" if max(ratios) > 1.001 else "attenuating"
        print(f"h = {h:.3f} s   ({status})")
        print("   ||e_i||_2 :", "  ".join(f"{n:6.3f}" for n in norms))
        print("   ratios    :", "  ".join(f"{r:6.3f}" for r in ratios))
        print()


# =============================================================== PART B
def part_B(h=1.0):
    print("=" * 68)
    print(f"PART B — empirical |G(jw)| vs analytic, at h = {h} s")
    print("=" * 68)
    print(f"  {'omega':>8s} {'empirical':>12s} {'analytic':>12s} {'err %':>8s}")

    for omega in [0.1, 0.3, 0.5, 0.8, 1.2, 2.0, 3.0]:
        t, e = simulate(h, n_veh=4, t_end=200.0, pert="sine",
                        omega=omega, amp=0.5)
        # steady-state amplitude ratio between consecutive vehicles
        mask = t > 120.0
        a1 = np.abs(e[mask, 1]).max()
        a2 = np.abs(e[mask, 2]).max()
        emp = a2 / a1
        ana = G_magnitude(omega, h)
        print(f"  {omega:8.2f} {emp:12.4f} {ana:12.4f} "
              f"{abs(emp-ana)/ana*100:8.2f}")
    print()


# =============================================================== PART C
def part_C():
    print("=" * 68)
    print("PART C — THE MAIN RESULT: peak gain vs time gap h")
    print("=" * 68)
    print(f"  Predicted threshold:  h* = sqrt(2/kp) = {H_CRIT:.4f} s")
    print(f"  Below h*, the platoon should amplify.  Above, attenuate.\n")
    print(f"  {'h (s)':>8s} {'||G||_inf':>12s} {'verdict':>14s}")

    hs = [0.5, 1.0, 1.2, 1.4, H_CRIT - 0.05, H_CRIT, H_CRIT + 0.05, 1.8, 2.5]
    for h in sorted(hs):
        pk = G_peak(h)
        verdict = "AMPLIFIES" if pk > 1.0 + 1e-9 else "attenuates"
        mark = "  <-- threshold" if abs(h - H_CRIT) < 1e-6 else ""
        print(f"  {h:8.4f} {pk:12.5f} {verdict:>14s}{mark}")

    # locate the crossing numerically by bisection
    lo, hi = 0.1, 5.0
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if G_peak(mid) > 1.0:
            lo = mid
        else:
            hi = mid
    print(f"\n  Numerically located crossing : h = {0.5*(lo+hi):.6f} s")
    print(f"  Theoretical prediction       : h = {H_CRIT:.6f} s")
    print(f"  Agreement                    : "
          f"{abs(0.5*(lo+hi)-H_CRIT)/H_CRIT*100:.4f} % error")

    # and confirm the h = 0 impossibility result
    print(f"\n  Constant-spacing check (h = 0): ||G||_inf = {G_peak(0.0):.4f}"
          f"  -> {'UNSTABLE, as Swaroop & Hedrick proved' if G_peak(0.0) > 1 else '??'}")


if __name__ == "__main__":
    part_A()
    part_B()
    part_C()

PART A — spacing error propagating along the platoon
kp = 0.8, kd = 1.2   ->   predicted threshold h* = sqrt(2/kp) = 1.5811 s

h = 0.800 s   (AMPLIFYING)
   ||e_i||_2 :  0.610   0.597   0.592   0.597   0.608   0.623   0.641   0.662
   ratios    :  0.978   0.993   1.008   1.018   1.024   1.029   1.033

h = 1.581 s   (attenuating)
   ||e_i||_2 :  0.487   0.404   0.352   0.318   0.293   0.274   0.259   0.247
   ratios    :  0.830   0.872   0.903   0.922   0.936   0.945   0.952

h = 2.500 s   (attenuating)
   ||e_i||_2 :  0.391   0.280   0.221   0.185   0.160   0.143   0.129   0.118
   ratios    :  0.717   0.788   0.837   0.868   0.889   0.905   0.916

PART B — empirical |G(jw)| vs analytic, at h = 1.0 s
     omega    empirical     analytic    err %
      0.10       1.0070       1.0070     0.00
      0.30       1.0320       1.0321     0.02
      0.50       0.9707       0.9701     0.06
      0.80       0.7325       0.7301     0.33
      1.20       0.4913       0.4886     0.56
      2.00    